# 07 - SHAP Analysis

Explain model predictions using SHAP values for the best model.

## Objectives:
- Load the saved best model
- Generate SHAP values
- Analyze global feature importance
- Explain individual predictions
- Create visualizations (summary, waterfall, force plots)

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from src.forecasting import DemandForecaster
from src.explainability import ModelExplainer
from src.utils import load_dataframe

import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

## 1. Load Best Model and Data

In [ ]:
forecaster = DemandForecaster()
forecaster.load_model('../models/best_model.joblib')

df = load_dataframe('../data/processed/engineered_features.csv')
df['date'] = pd.to_datetime(df['date'])

print(f"Model type: {forecaster.model_type}")
print(f"Data shape: {df.shape}")
print(f"Features: {len(forecaster.feature_names)}")

## 2. Prepare Sample Data

SHAP computation is slow on large datasets, so we use a representative sample.

In [ ]:
sample_size = 1000
df_sample = df.sample(n=min(sample_size, len(df)), random_state=42)
X_sample, y_sample = forecaster.prepare_features(df_sample, target_col='sales')

print(f"Sample size: {len(X_sample)}")
print(f"Number of features: {X_sample.shape[1]}")

## 3. Create SHAP Explainer

In [ ]:
explainer = ModelExplainer(forecaster.model, forecaster.feature_names)
explainer.create_explainer(X_sample, explainer_type='tree')

print("SHAP explainer created!")

## 4. Calculate SHAP Values

In [ ]:
print("Calculating SHAP values... (may take a few minutes)")
shap_values = explainer.calculate_shap_values(X_sample)

print(f"\nSHAP values calculated! Shape: {shap_values.shape}")

## 5. Global Feature Importance (by SHAP)

In [ ]:
top_features = explainer.get_top_features(X_sample, top_n=20)

print("Top 20 Most Important Features (by SHAP):")
print(top_features.to_string(index=False))

# Save for dashboard use
top_features.to_csv('../reports/exports/shap_feature_importance.csv', index=False)
print("\nSaved to reports/exports/shap_feature_importance.csv")

## 6. SHAP Summary Plot

In [ ]:
plt.figure(figsize=(12, 8))
explainer.plot_summary(X_sample, max_display=20)
plt.tight_layout()
plt.savefig('../reports/figures/shap_summary.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to reports/figures/shap_summary.png")

## 7. Feature Importance Bar Plot

In [ ]:
plt.figure(figsize=(12, 8))
explainer.plot_feature_importance(X_sample, max_display=20)
plt.tight_layout()
plt.savefig('../reports/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()
print("Saved to reports/figures/feature_importance.png")

## 8. Explain a Single Prediction

In [ ]:
instance_idx = 0
explanation = explainer.explain_prediction(X_sample, instance_idx)

print(f"Prediction: {explanation['prediction']:.2f}")
print(f"Base value: {explanation['base_value']:.2f}")
print(f"\nTop 10 Contributing Features:")
for i, (feature, info) in enumerate(list(explanation['shap_values'].items())[:10], 1):
    print(f"{i:2d}. {feature:35s} = {info['value']:8.2f}  (SHAP: {info['shap_value']:+8.2f})")

## 9. Waterfall Plot

In [ ]:
plt.figure(figsize=(12, 8))
explainer.plot_waterfall(X_sample, instance_index=instance_idx)
plt.tight_layout()
plt.show()

## 10. Force Plot

In [ ]:
plt.figure(figsize=(14, 4))
explainer.plot_force(X_sample, instance_index=instance_idx)
plt.tight_layout()
plt.show()

## 11. Key Insights

In [ ]:
print("=" * 60)
print("KEY INSIGHTS FROM SHAP ANALYSIS")
print(f"Model: {forecaster.model_type}")
print("=" * 60)

print("\n1. Most Important Features:")
for i, row in top_features.head(5).iterrows():
    print(f"   {i+1}. {row['feature']} (importance: {row['importance']:.4f})")

print("\n2. Interpretation:")
print("   - Positive SHAP = feature pushes prediction higher")
print("   - Negative SHAP = feature pushes prediction lower")
print("   - Red = high feature value, Blue = low feature value")

print("\n3. Business Use Cases:")
print("   - Understand what drives demand for each product")
print("   - Validate model is not using spurious patterns")
print("   - Build trust with stakeholders via transparent predictions")
print("=" * 60)

## Summary

SHAP analysis completed:
- SHAP explainer created for best model
- SHAP values calculated
- Global feature importance analyzed
- Individual predictions explained with waterfall + force plots
- Visualizations saved to reports/

The model is now fully explainable and ready for deployment!